# Milestone 4 — Best model & error analysis

Same split, same target (`diagnosis_1`, 3 classes), same metric (macro-F1) as Milestones 2 and 3.

1. ResNet-18 vs ViT-B/16, both frozen, same head (Session 9's fair comparison)
2. The best model: pick on validation, use the test set once
3. Error analysis
4. Fairness check and the final scoreboard

In Colab: **Runtime → Change runtime type → T4 GPU** before running.

## 0. Setup

In [ ]:
# PURPOSE: get my repo and the dataset into Colab (skip this cell on my laptop).
# HOW: clone the repo, then the same download cell as Session 9.

import os, urllib.request, zipfile
from pathlib import Path

!git clone -q https://github.com/sopipi180/ComputerVisionandSpeechRecognition.git
%cd ComputerVisionandSpeechRecognition

DEST = Path("/content/milk10k")
if not (DEST / "metadata.csv").exists():
    urllib.request.urlretrieve(
        "https://isic-archive.s3.amazonaws.com/dois/10.34970-648456/milk10k.zip",
        "/content/milk10k.zip")
    with zipfile.ZipFile("/content/milk10k.zip") as z:
        z.extractall(DEST)

os.environ["MILK10K_DIR"] = str(DEST)      # common/config.py reads this

In [ ]:
# PURPOSE: libraries, my repo's config, and CPU or GPU.

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score, ConfusionMatrixDisplay
from torchvision.models import resnet18, ResNet18_Weights, vit_b_16, ViT_B_16_Weights

# My repo: walk up until common/config.py is found (as in Sessions 6 and 7)
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "common" / "config.py").is_file())
sys.path.insert(0, str(ROOT))
from common import config

CACHE = ROOT / "m4_best_model" / "_cache"      # decoded images and features (not in git)
CACHE.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(config.SEED)
print(device)

## 1. Data — my split, as in Milestones 2 and 3

The split CSVs are grouped by lesion, so both photos of a lesion stay on the same side.
One dermoscopic image per lesion, as in Sessions 5–7.

In [ ]:
# PURPOSE: load MY split and turn the labels into numbers.

CLASSES = config.CLASSES_3                     # ["Benign", "Indeterminate", "Malignant"]

splits = {}
for name in ["train", "val", "test"]:
    df = pd.read_csv(config.SPLIT_DIR / f"{name}.csv")
    splits[name] = df[df["image_type"].str.startswith("derm")].reset_index(drop=True)

y = {name: torch.tensor(df["diagnosis_1"].map(CLASSES.index).to_numpy()) for name, df in splits.items()}

for name, df in splits.items():
    print(name, len(df), df["diagnosis_1"].value_counts().to_dict())     # 3667 / 786 / 787

In [ ]:
# PURPOSE: decode every image once at 224x224 and keep it (Session 7 / Session 9).
# HOW: if the saved file exists, load it; if not, build it image by image and save it.

def load_images(df, name):
    f = CACHE / f"images_{name}.pt"
    if not f.exists():
        X = torch.stack([torch.from_numpy(np.array(Image.open(config.image_path(i)).convert("RGB").resize((224, 224))))
                         .permute(2, 0, 1) for i in df["isic_id"]])          # (H,W,3) -> (3,H,W)
        torch.save(X, f)
    return torch.load(f)

X = {name: load_images(df, name) for name, df in splits.items()}
print({name: tuple(x.shape) for name, x in X.items()})

In [ ]:
# PURPOSE: scale the pixels like ImageNet, because both networks learned on ImageNet (Session 9).

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def prep(xb):
    return (xb.float() / 255 - MEAN) / STD

## 2. ResNet-18 vs ViT-B/16, a fair comparison

Session 9's recipe: both backbones **frozen**, every lesion turned into one vector once
(ResNet-18: 512 numbers, ViT: 768), then the **same** head trained on top.
Only the backbone changes. This time on all 3,667 training lesions.

In [ ]:
# PURPOSE: turn every lesion into one vector per backbone (Session 9, run on the GPU).
# HOW: remove each network's last layer, run all images once without gradients, save the result.

def backbone(name):
    if name == "resnet18":
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        net.fc = nn.Identity()                    # output: 512 numbers instead of 1000 classes
    else:
        net = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        net.heads = nn.Identity()                 # output: the class token, 768 numbers
    return net.eval().to(device)

def features(name, X, split):
    f = CACHE / f"features_{name}_{split}.pt"
    if not f.exists():
        net, t0 = backbone(name), time.perf_counter()
        with torch.no_grad():
            out = torch.cat([net(prep(X[i:i + 64]).to(device)).cpu() for i in range(0, len(X), 64)])
        torch.save(out, f)
        print(f"{name} {split}: {tuple(out.shape)} in {time.perf_counter() - t0:.0f} s")
    return torch.load(f)

feats = {name: {split: features(name, X[split], split) for split in ["train", "val", "test"]}
         for name in ["resnet18", "vit_b_16"]}

In [ ]:
# PURPOSE: the same head for both backbones (my Session 9 Exercise 4).

def train_head(F_train, y, n_classes=3, epochs=25):
    torch.manual_seed(0)
    head = nn.Linear(F_train.shape[1], n_classes)                      # the only weights we train
    class_weights = len(y) / (n_classes * torch.bincount(y, minlength=n_classes).float())
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)                # rare classes count more (Session 6)
    optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
    for epoch in range(epochs):
        order = torch.randperm(len(y))                                 # new random order in each epoch
        for i in range(0, len(y), 64):
            idx = order[i:i + 64]                                      # one batch of 64 lesions
            optimizer.zero_grad()
            loss = loss_fn(head(F_train[idx]), y[idx])
            loss.backward()
            optimizer.step()
    return head

def macro_f1(head, F_, y):
    with torch.no_grad():
        pred = head(F_).argmax(1)
    return f1_score(y, pred, average="macro")

In [ ]:
# PURPOSE: train both heads and score them on the same validation lesions. Test stays closed.

heads, val_f1 = {}, {}
for name in ["resnet18", "vit_b_16"]:
    heads[name] = train_head(feats[name]["train"], y["train"])
    val_f1[name] = macro_f1(heads[name], feats[name]["val"], y["val"])
    print(f"{name:9s} validation macro-F1 = {val_f1[name]:.3f}")

In [ ]:
# PURPOSE: where does each backbone make its mistakes? (Session 9, section 4.2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ["resnet18", "vit_b_16"]):
    with torch.no_grad():
        pred = heads[name](feats[name]["val"]).argmax(1)
    ConfusionMatrixDisplay.from_predictions(y["val"], pred, display_labels=CLASSES, cmap="Oranges", ax=ax, colorbar=False)
    ax.set_title(f"{name}: macro-F1 {val_f1[name]:.3f}")
plt.tight_layout()